In [ ]:
#5. Implement the Continuous Bag of Words (CBOW) Model. Stages can be: 
a. Data preparation 
b. Generate training data 
c. Train model 
d. Output  

In [1]:
#Import Necessary Libraries

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Lambda, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# a. Data preparation

In [2]:
# Sample corpus

corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are great pets",
    "the mat is soft and warm"
]

In [3]:
# Tokenize and lowercase the text

tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1

In [4]:
# Convert text to sequences of integers

sequences = tokenizer.texts_to_sequences(corpus)

print("Vocabulary:", tokenizer.word_index)
print("Total Words (including padding):", total_words)
print("\nSequences:")
for seq in sequences:
    print(seq)

Vocabulary: {'the': 1, 'sat': 2, 'on': 3, 'mat': 4, 'and': 5, 'cat': 6, 'dog': 7, 'log': 8, 'cats': 9, 'dogs': 10, 'are': 11, 'great': 12, 'pets': 13, 'is': 14, 'soft': 15, 'warm': 16}
Total Words (including padding): 17

Sequences:
[1, 6, 2, 3, 1, 4]
[1, 7, 2, 3, 1, 8]
[9, 5, 10, 11, 12, 13]
[1, 4, 14, 15, 5, 16]


In [ ]:
# b. Generate training data

In [5]:
def generate_training_data(sequences, window_size=2):
    contexts = []
    targets = []
    
    for sequence in sequences:
        for i in range(window_size, len(sequence) - window_size):
            context = sequence[i - window_size:i] + sequence[i + 1:i + window_size + 1]
            target = sequence[i]
            contexts.append(context)
            targets.append(target)
    
    return np.array(contexts), np.array(targets)

X, y = generate_training_data(sequences, window_size=2)

In [6]:
# Pad context sequences for consistent input shape

X = pad_sequences(X, maxlen=4, padding='pre')

print("\nContext Samples (X):\n", X[:5])
print("Target Samples (y):\n", y[:5])


Context Samples (X):
 [[ 1  6  3  1]
 [ 6  2  1  4]
 [ 1  7  3  1]
 [ 7  2  1  8]
 [ 9  5 11 12]]
Target Samples (y):
 [ 2  3  2  3 10]


In [ ]:
# c. Train model

In [7]:
# Define the CBOW architecture

model = Sequential()
model.add(Embedding(input_dim=total_words, output_dim=10, input_length=4))
model.add(Lambda(lambda x: tf.reduce_mean(x, axis=1)))  # Average context embeddings
model.add(Dense(total_words, activation='softmax'))

In [8]:
# Compile the model

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [9]:
# Train the model

print("\nTraining the CBOW model...\n")
model.fit(X, y, epochs=100, verbose=0)
print("Training complete.\n")


Training the CBOW model...

Training complete.



In [ ]:
# d. Output

In [10]:
# Extract learned word embeddings

word_embeddings = model.layers[0].get_weights()[0]

In [11]:
# Create mapping of words to their embeddings

word_index = tokenizer.word_index
embeddings_dict = {word: word_embeddings[idx] for word, idx in word_index.items()}

In [12]:
# Display results

print("Vocabulary Size:", len(word_index))
print("Sample Vocabulary:", list(word_index.items())[:10])
print("\nWord Embeddings:\n")
print("{:<10} | {}".format("Word", "Embedding Vector"))
print("-" * 60)
for word, embedding in embeddings_dict.items():
    print("{:<10} | {}".format(word, np.round(embedding, 3)))

Vocabulary Size: 16
Sample Vocabulary: [('the', 1), ('sat', 2), ('on', 3), ('mat', 4), ('and', 5), ('cat', 6), ('dog', 7), ('log', 8), ('cats', 9), ('dogs', 10)]

Word Embeddings:

Word       | Embedding Vector
------------------------------------------------------------
the        | [-0.11   0.129  0.112  0.081 -0.065  0.165  0.09  -0.1    0.137  0.083]
sat        | [ 0.098  0.109 -0.081  0.157  0.099 -0.109  0.126 -0.058  0.109  0.109]
on         | [-0.084 -0.06   0.093  0.105 -0.094  0.144  0.146 -0.052  0.135  0.128]
mat        | [0.106 0.121 0.078 0.118 0.149 0.011 0.133 0.058 0.082 0.087]
and        | [-0.048  0.128  0.153 -0.122  0.071  0.071  0.141  0.074 -0.107 -0.086]
cat        | [ 0.167  0.137 -0.059  0.092 -0.049  0.035  0.157 -0.129  0.083  0.129]
dog        | [ 0.153  0.119 -0.061  0.139 -0.045  0.095  0.127 -0.111  0.125  0.081]
log        | [ 0.098  0.161 -0.139  0.189  0.095 -0.094  0.062 -0.124  0.09   0.098]
cats       | [-0.08   0.067  0.108 -0.113 -0.149 -0.006  0

In [ ]:
ThankYou!!